In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
import os

data_path = "/content/drive/MyDrive/Colab_Notebooks/Teknofest"
os.chdir(data_path)
print(os.listdir("."))


['test.csv', 'train.csv', 'adresler_koordinatli.csv', 'Untitled0.ipynb']


In [23]:
import pandas as pd

train_data = pd.read_csv("train.csv")

In [24]:
train_data.head()

,address,label
0,Akarca Mah. Adnan Menderes Cad. 864.Sok. No:15 D.1 K.2 .,8831
1,Cumhuriye Mah. Hükümet Cad. Sivriler İşhanı No:3 Fethiye/Muğla Foto Kandiye Muğla / Fethiye,8810
2,İsmet inönü mahallesi 2001 sokak no:2 Çeşme belediyesi Çeşme,3067
3,"Gazeteci Hasan Tahsin Caddesi, No:10/3, Gizem Apartman",8210
4,Bitez mahallesi Adnan Menderes caddesi gündonumu mevkii 1410.sokak no 90A Doria hotel,9675


In [25]:
import pandas as pd

# Farklı label değerleri
unique_labels = train_data["label"].unique()
print("Farklı label sayısı:", train_data["label"].nunique())
print("Label değerleri:", unique_labels)


Farklı label sayısı: 10390
Label değerleri: [8831 8810 3067 ... 1887 2069 1832]


In [26]:
train_data = train_data[:100000]

In [36]:
import re
import pandas as pd

# Tüm satırların ve sütunların tam görünmesi için pandas ayarları
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

def normalize_address_final(addr: str):
    addr = addr.lower()

    # YENİ ÖN İŞLEME ADIMI: Sayı ve kelime arasına sıkışmış noktaları temizle
    # '864.sok' -> '864 sok', '1675.cadde' -> '1675 cadde' gibi dönüşümleri yapar.
    # Bu, sonraki kısaltma açma adımlarının doğru çalışmasını sağlar.
    # (\d+): Bir veya daha fazla sayıyı yakala (Grup 1).
    # \.   : Hemen ardından gelen noktayı bul.
    # r'\1 ': Yakalanan sayıyı (Grup 1) geri yaz ve bir boşluk ekle.
    addr = re.sub(r'(\d+)\.', r'\1 ', addr)

    # 1. Kısaltmaları aç
    addr = re.sub(r"\bmah\.?\b", " mahallesi", addr)
    addr = re.sub(r"\bcad\.?\b", " caddesi", addr)
    addr = re.sub(r"\bsok\.?\b", " sokak", addr)
    addr = re.sub(r"\bapt\.?\b", " apartmanı", addr)
    addr = re.sub(r"\bbul\.?\b", " bulvar", addr)

    # 2. Apartmanla ilgili bilgileri temizle
    addr = re.sub(r'(\b\w+\s+)?\b\w+\s+apartman[ıi]?\b', ' ', addr)

    # 3. no, kat, daire gibi spesifik bilgileri temizle
    pattern = r'[\s:./-]*\d+([a-zA-Z]|\s?[/\-]\s?[a-zA-Z0-9]+)?'
    addr = re.sub(r'\b(no)' + pattern, ' ', addr)
    addr = re.sub(r'\b(kat|k)\b' + pattern, ' ', addr)
    addr = re.sub(r'\b(daire|d)\b' + pattern, ' ', addr)

    # 4. Anahtar kelime olmadan tek başına duran kapı numaralarını temizleme
    addr = re.sub(r'\b\d+[/\-]\s?[a-zA-Z]\b', ' ', addr) # 50/A, 50-A
    addr = re.sub(r'\b\d+\s+[a-zA-Z]\b', ' ', addr)     # 50 A

    # 5. Geriye kalan genel noktalama işaretlerini temizleme
    addr = re.sub(r"[,.;:!?()\[\]{}|<>\-_=+*/\\]", " ", addr)

    # 6. Akıllı sayı silme (sokak, cadde vb. isimlerini koruyarak)
    addr = re.sub(r'\b\d+\b(?!\s*(sokak|cadde|mahalle|bulvar|yıl))', ' ', addr) # 'yıl' da koruma listesine eklendi

    # 7. Fazla boşlukları birleştir ve kırp
    addr = re.sub(r"\s+", " ", addr).strip()

    return addr


# DataFrame üzerinde normalize etme
train_data['normalized_address'] = train_data['address'].apply(normalize_address_final)
print(train_data[['normalized_address', 'label']].head())

                                                                              normalized_address  \
0                                              akarca mahallesi adnan menderes caddesi 864 sokak   
1  cumhuriye mahallesi hükümet caddesi sivriler i̇şhanı fethiye muğla foto kandiye muğla fethiye   
2                                       i̇smet inönü mahallesi 2001 sokak çeşme belediyesi çeşme   
3                                                                  gazeteci hasan tahsin caddesi   
4                 bitez mahallesi adnan menderes caddesi gündonumu mevkii 1410 sokak doria hotel   

   label  
0   8831  
1   8810  
2   3067  
3   8210  
4   9675  
